In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RPKM_DIR = Path('/cta/users/guneyn23/diffbind/no_summits/differential_peak_center_20kb_100windows/rpkm')
OUTPUT_DIR = Path('/cta/users/guneyn23/diffbind/no_summits/differential_peak_center_20kb_100windows/plots')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PEAK_GROUPS = {
    '4h_noUV_specific': 'no_summits_noUV_vs_UV_4h_DESeq2_noUV_higher_FDR005_100windows',
    '4h_specific': 'no_summits_noUV_vs_UV_4h_DESeq2_UV_4h_higher_FDR005_100windows',
    '8h_noUV_specific': 'no_summits_noUV_vs_UV_8h_DESeq2_noUV_higher_FDR005_100windows',
    '8h_specific': 'no_summits_noUV_vs_UV_8h_DESeq2_UV_8h_higher_FDR005_100windows',
}

DAMAGE_FILES = {
    ('64', '1m'): 'R3Hela_1m64_CGATGT_S5_hg38_primary_assembly_DS',
    ('64', '15m'): 'R3Hela_15m64_TTAGGC_S1_hg38_primary_assembly_DS',
    ('64', '30m'): 'R3Hela_30m64_TGACCA_S7_hg38_primary_assembly_DS',
    ('64', '1h'): 'R3Hela_1h64_ACAGTG_S3_hg38_primary_assembly_DS',
    ('64', '4h'): 'R3Hela_4h64_GCCAAT_S9_hg38_primary_assembly_DS',
    ('64', '8h'): 'R3Hela_8h64_CAGATC_S11_hg38_primary_assembly_DS',
    ('CPD', '1m'): 'R3Hela_1mCPD_GATCAG_S6_hg38_primary_assembly_DS',
    ('CPD', '15m'): 'R3Hela_15mCPD_TAGCTT_S2_hg38_primary_assembly_DS',
    ('CPD', '30m'): 'R3Hela_30mCPD_GGCTAC_S8_hg38_primary_assembly_DS',
    ('CPD', '1h'): 'R3Hela_1hCPD_CTTGTA_S4_hg38_primary_assembly_DS',
    ('CPD', '4h'): 'R3Hela_4hCPD_AGTCAA_S10_hg38_primary_assembly_DS',
    ('CPD', '8h'): 'R3Hela_8hCPD_AGTTCC_S12_hg38_primary_assembly_DS',
}

TIME_POINTS = ['1m', '15m', '30m', '1h', '4h', '8h']
DAMAGE_TYPES = ['CPD', '64']
SMOOTHING_WINDOWS = 5  # 5 windows x 200 bp = 1 kb
COLORS = {
    '1m': '#56B4E9', '15m': '#0072B2', '30m': '#E69F00',
    '1h': '#009E73', '4h': '#D55E00', '8h': '#CC79A7',
}
DAMAGE_LABELS = {'CPD': 'CPD', '64': '6-4PP'}
PEAK_LABELS = {
    '4h_noUV_specific': '4h noUV-specific peaks',
    '4h_specific': '4h-specific peaks',
    '8h_noUV_specific': '8h noUV-specific peaks',
    '8h_specific': '8h-specific peaks',
}

In [ ]:
def mean_profile(file_path):
    data = pd.read_csv(file_path, sep='\t', header=None)
    rpkm = pd.to_numeric(data.iloc[:, -1])
    data['window'] = pd.to_numeric(
        data.iloc[:, 3].astype(str).str.rsplit('_', n=1).str[-1]
    )
    data['rpkm'] = rpkm
    profile = data.groupby('window', sort=True)['rpkm'].mean()
    return profile.reindex(range(1, 101))

profiles = []

for peak_group, peak_file_stem in PEAK_GROUPS.items():
    for damage_type in DAMAGE_TYPES:
        for time_point in TIME_POINTS:
            damage_file_stem = DAMAGE_FILES[(damage_type, time_point)]
            real_file = RPKM_DIR / f'{damage_file_stem}_{peak_file_stem}_rpkm.bed'
            simulated_file = RPKM_DIR / f'{damage_file_stem}_sim_{peak_file_stem}_rpkm.bed'
            real_raw = mean_profile(real_file)
            simulated_raw = mean_profile(simulated_file)
            real_profile = real_raw.rolling(
                SMOOTHING_WINDOWS, center=True, min_periods=1
            ).mean()
            simulated_profile = simulated_raw.rolling(
                SMOOTHING_WINDOWS, center=True, min_periods=1
            ).mean()
            ratio_profile = real_profile / simulated_profile.replace(0, float('nan'))

            for window in range(1, 101):
                profiles.append({
                    'peak_group': peak_group,
                    'damage_type': damage_type,
                    'time_point': time_point,
                    'window': window,
                    'distance_kb': -10 + (window - 0.5) * 0.2,
                    'mean_real_rpkm_raw': real_raw.loc[window],
                    'mean_simulated_rpkm_raw': simulated_raw.loc[window],
                    'mean_real_rpkm': real_profile.loc[window],
                    'mean_simulated_rpkm': simulated_profile.loc[window],
                    'real_sim_ratio': ratio_profile.loc[window],
                })

summary = pd.DataFrame(profiles)
summary.to_csv(OUTPUT_DIR / 'mean_rpkm_profiles.tsv', sep='\t', index=False)
summary.head()

In [ ]:
# Plot real RPKM profiles
REAL_YLIM = {
    damage_type: (
        0.1,
        0.5,
    )
    for damage_type in DAMAGE_TYPES
}
print('Real y-limits:', REAL_YLIM)

for peak_group in PEAK_GROUPS:
    for damage_type in DAMAGE_TYPES:
        selected = summary[(summary['peak_group'] == peak_group) & (summary['damage_type'] == damage_type)]
        fig, ax = plt.subplots(figsize=(10, 5))

        for time_point in TIME_POINTS:
            values = selected[selected['time_point'] == time_point]
            ax.plot(values['distance_kb'], values['mean_real_rpkm'], color=COLORS[time_point], linewidth=2, label=time_point)

        ax.axvline(0, color='gray', linestyle='--', linewidth=1)
        ax.set_xlim(-10, 10)
        ax.set_ylim(*REAL_YLIM[damage_type])
        ax.set_xticks([-10, -5, 0, 5, 10])
        ax.set_xlabel('Distance from differential peak center (kb)')
        ax.set_ylabel('Mean real damage signal (RPKM)')
        ax.set_title(f'Real {DAMAGE_LABELS[damage_type]} profile: {PEAK_LABELS[peak_group]}')
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(title='Time after UV')
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / f'{peak_group}_{damage_type}_real.png', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(fig)

In [ ]:
# Plot simulated RPKM profiles
SIMULATED_YLIM = {
    damage_type: (
        0.1,
        0.5,
    )
    for damage_type in DAMAGE_TYPES
}
print('Simulated y-limits:', SIMULATED_YLIM)

for peak_group in PEAK_GROUPS:
    for damage_type in DAMAGE_TYPES:
        selected = summary[(summary['peak_group'] == peak_group) & (summary['damage_type'] == damage_type)]
        fig, ax = plt.subplots(figsize=(10, 5))

        for time_point in TIME_POINTS:
            values = selected[selected['time_point'] == time_point]
            ax.plot(values['distance_kb'], values['mean_simulated_rpkm'], color=COLORS[time_point], linewidth=2, label=time_point)

        ax.axvline(0, color='gray', linestyle='--', linewidth=1)
        ax.set_xlim(-10, 10)
        ax.set_ylim(*SIMULATED_YLIM[damage_type])
        ax.set_xticks([-10, -5, 0, 5, 10])
        ax.set_xlabel('Distance from differential peak center (kb)')
        ax.set_ylabel('Mean simulated damage signal (RPKM)')
        ax.set_title(f'Simulated {DAMAGE_LABELS[damage_type]} profile: {PEAK_LABELS[peak_group]}')
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(title='Time after UV')
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / f'{peak_group}_{damage_type}_simulated.png', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(fig)

In [ ]:
# Plot real/simulated RPKM ratios
RATIO_YLIM = {
    damage_type: (
        0.4,
        1.75,
    )
    for damage_type in DAMAGE_TYPES
}
print('Real/sim y-limits:', RATIO_YLIM)

for peak_group in PEAK_GROUPS:
    for damage_type in DAMAGE_TYPES:
        selected = summary[(summary['peak_group'] == peak_group) & (summary['damage_type'] == damage_type)]
        fig, ax = plt.subplots(figsize=(10, 5))

        for time_point in TIME_POINTS:
            values = selected[selected['time_point'] == time_point]
            ax.plot(values['distance_kb'], values['real_sim_ratio'], color=COLORS[time_point], linewidth=2, label=time_point)

        ax.axhline(1, color='black', linestyle='--', linewidth=1)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)
        ax.set_xlim(-10, 10)
        ax.set_ylim(*RATIO_YLIM[damage_type])
        ax.set_xticks([-10, -5, 0, 5, 10])
        ax.set_xlabel('Distance from differential peak center (kb)')
        ax.set_ylabel('Real / simulated mean RPKM')
        ax.set_title(f'Real/simulated {DAMAGE_LABELS[damage_type]} profile: {PEAK_LABELS[peak_group]}')
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(title='Time after UV')
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / f'{peak_group}_{damage_type}_real_sim_ratio.png', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(fig)